# Curate a cluster with the CuratorAgent

Playground for the deck curation agent. It runs against a **copy**
of the live collection, and the agent can only *propose* changes —
nothing is applied to the collection (even the copy) unless you
explicitly call `repository.update/add/remove` afterwards.

Requires: an OpenAI-compatible server running (e.g. vLLM) with the
model configured below.

In [ ]:
from pathlib import Path

COLLECTION_PATH = Path(
    "/home/gianluca/.local/share/Anki2/User 1/collection.anki2"
)
LLM_HOST = "localhost"
LLM_PORT = "8000"
LLM_MODEL = "qwen3"  # name the server was started with

SEED_QUERY = "beta_2 adam"
INSTRUCTION = None  # e.g. "focus on removing duplication"

In [ ]:
import shutil
import tempfile

from anki.collection import Collection

tmp_dir = Path(tempfile.mkdtemp(prefix="anki_curator_"))
copy_path = tmp_dir / "collection.anki2"
shutil.copy(COLLECTION_PATH, copy_path)
col = Collection(str(copy_path))
print(f"Opened a copy of the collection at {copy_path}")

In [ ]:
from addon.application.services.curator_agent import CuratorAgent
from addon.application.services.curator_tools import CuratorTools
from addon.infrastructure.configuration.settings import AddonConfig
from addon.infrastructure.external_services.openai import (
    OpenAIClient,
)
from addon.infrastructure.persistence.anki_note_repository import (
    AnkiNoteRepository,
)


class DictConfig:
    def __init__(self, config: dict):
        self._config = config

    def getConfig(self, module: str):
        return self._config


config = AddonConfig(
    DictConfig(
        {
            "openai_host": LLM_HOST,
            "openai_port": LLM_PORT,
            "openai_model": LLM_MODEL,
            "openai_max_tokens": 4096,
            "openai_reasoning": True,
        }
    )
)
client = OpenAIClient(config)
repository = AnkiNoteRepository(col)
tools = CuratorTools(repository)
agent = CuratorAgent(client, tools, max_steps=15)

In [ ]:
seed_ids = repository.search(SEED_QUERY, limit=5)
for nid in seed_ids:
    print(tools.read_note(nid))
    print("-" * 79)
seed_id = seed_ids[0]

Run the agent. This makes one LLM call per step (up to
`max_steps`), so it can take a few minutes with a reasoning model.

In [ ]:
%%time
session = agent.run(seed_id, instruction=INSTRUCTION)

In [ ]:
import json

for msg in session.transcript[2:]:  # skip system prompt and seed
    if msg["role"] == "assistant":
        try:
            step = json.loads(msg["content"])
            print(f"THOUGHT: {step['thought']}")
            print(f"ACTION:  {json.dumps(step['action'])}")
        except json.JSONDecodeError:
            print(f"INVALID MODEL OUTPUT: {msg['content'][:200]}")
    else:
        print(f"OBSERVATION:\n{msg['content']}")
    print("-" * 79)
print(f"SUMMARY: {session.summary}")

In [ ]:
from addon.domain.entities.proposals import (
    CreateProposal,
    DeleteProposal,
    EditProposal,
)

for p in session.change_set:
    if isinstance(p, EditProposal):
        print(f"EDIT note {p.note_id} - {p.rationale}")
        print(f"  before front: {p.before.front}")
        print(f"  after front:  {p.after.front}")
    elif isinstance(p, CreateProposal):
        print(f"CREATE - {p.rationale}")
        print(f"  front: {p.note.front}")
        print(f"  back:  {p.note.back}")
    elif isinstance(p, DeleteProposal):
        print(f"DELETE note {p.note_id} - {p.rationale}")
        print(f"  front: {p.before.front}")
    print("-" * 79)
print(f"{len(session.change_set)} proposal(s)")

In [ ]:
col.close()
shutil.rmtree(tmp_dir)